In [1]:
import tensorflow as tf

shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

In [2]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [3]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character",
 standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

In [4]:
encoded -= 2 # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2 # number of distinct chars = 39
dataset_size = len(encoded) # total number of chars = 1,115,394

In [5]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [6]:
length = 100
tf.random.set_seed(42)

train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True,
 seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

In [ ]:
model = tf.keras.Sequential([
 tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
 tf.keras.layers.GRU(128, return_sequences=True),
 tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
 metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
 "my_shakespeare_model", monitor="val_accuracy", save_best_only=True)

history = model.fit(train_set, validation_data=valid_set, epochs=10,
 callbacks=[model_ckpt])

In [ ]:
shakespeare_model = tf.keras.Sequential([
 text_vec_layer,
 tf.keras.layers.Lambda(lambda X: X - 2), # no <PAD> or <UNK> tokens
 model
])

In [ ]:
y_proba = shakespeare_model.predict(["To be or not to b"])[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

ValueError: Unrecognized data type: x=['To be or not to b'] (of type <class 'list'>)

In [ ]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]]) # probas = 50%, 40%, and 10%
tf.random.set_seed(42)
tf.random.categorical(log_probas, num_samples=8) # draw 8 samples

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 1, 0, 2, 1, 0, 0, 1]], dtype=int64)>

In [ ]:
def next_char(text, temperature=1):
    y_proba = shakespeare_model.predict([text])[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
def extend_text(text, n_chars=50, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text


In [ ]:
tf.random.set_seed(42)
print(extend_text("To be or not to be", temperature=0.01))

ValueError: Unrecognized data type: x=['To be or not to be'] (of type <class 'list'>)

In [ ]:
print(extend_text("To be or not to be", temperature=1))

ValueError: Unrecognized data type: x=['To be or not to be'] (of type <class 'list'>)

In [ ]:
print(extend_text("To be or not to be", temperature=100))


ValueError: Unrecognized data type: x=['To be or not to be'] (of type <class 'list'>)

In [ ]:
def to_dataset_for_stateful_rnn(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(length + 1)).batch(1)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

stateful_train_set = to_dataset_for_stateful_rnn(encoded[:1_000_000], length)
stateful_valid_set = to_dataset_for_stateful_rnn(encoded[1_000_000:1_060_000],
 length)
stateful_test_set = to_dataset_for_stateful_rnn(encoded[1_060_000:], length)

In [ ]:
model = tf.keras.Sequential([
 tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16, batch_input_shape=[1, None]),
 tf.keras.layers.GRU(128, return_sequences=True, stateful=True),
 tf.keras.layers.Dense(n_tokens, activation="softmax")
])

ValueError: Unrecognized keyword arguments passed to Embedding: {'batch_input_shape': [1, None]}

In [ ]:
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
 metrics=["accuracy"])
history = model.fit(stateful_train_set, validation_data=stateful_valid_set,
 epochs=10, callbacks=[ResetStatesCallback(), model_ckpt])

NameError: name 'model_ckpt' is not defined

In [ ]:
import tensorflow_datasets as tfds

raw_train_set, raw_valid_set, raw_test_set = tfds.load(
 name="imdb_reviews",
 split=["train[:90%]", "train[90%:]", "test"],
 as_supervised=True
)

tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = raw_valid_set.batch(32).prefetch(1)
test_set = raw_test_set.batch(32).prefetch(1)

ModuleNotFoundError: No module named 'tensorflow_datasets'

In [ ]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))

NameError: name 'train_set' is not defined

In [ ]:
embed_size = 128

tf.random.set_seed(42)

model = tf.keras.Sequential([
 text_vec_layer,
 tf.keras.layers.Embedding(vocab_size, embed_size),
 tf.keras.layers.GRU(128),
 tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam",
 metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

NameError: name 'train_set' is not defined

In [ ]:
import os
import tensorflow_hub as hub
os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"
model = tf.keras.Sequential([
 hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
 trainable=True, dtype=tf.string, input_shape=[]),
 tf.keras.layers.Dense(64, activation="relu"),
 tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
 metrics=["accuracy"])
model.fit(train_set, validation_data=valid_set, epochs=10)

ModuleNotFoundError: No module named 'tensorflow_hub'

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets",
 extract=True)
text = (Path(path).with_name("spa-eng") / "spa.txt").read_text()

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 8s 3us/step


NameError: name 'Path' is not defined

In [ ]:
import numpy as np
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs) # separates the pairs into 2 lists


NameError: name 'text' is not defined

In [ ]:
for i in range(3):
    print(sentences_en[i], "=>", sentences_es[i])

In [ ]:
vocab_size = 1000
max_length = 50
text_vec_layer_en = tf.keras.layers.TextVectorization(
 vocab_size, output_sequence_length=max_length)
text_vec_layer_es = tf.keras.layers.TextVectorization(
 vocab_size, output_sequence_length=max_length)
text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])


In [ ]:
max_length = 50 # max length in the whole training set
embed_size = 128
pos_embed_layer = tf.keras.layers.Embedding(max_length, embed_size)
batch_max_len_enc = tf.shape(encoder_embeddings)[1]
encoder_in = encoder_embeddings + pos_embed_layer(tf.range(batch_max_len_enc))
batch_max_len_dec = tf.shape(decoder_embeddings)[1]
decoder_in = decoder_embeddings + pos_embed_layer(tf.range(batch_max_len_dec))

In [ ]:
N = 2 # instead of 6
num_heads = 8
dropout_rate = 0.1
n_units = 128 # for the first dense layer in each feedforward block
encoder_pad_mask = tf.math.not_equal(encoder_input_ids, 0)[:, tf.newaxis]
Z = encoder_in
for _ in range(N):
 skip = Z
 attn_layer = tf.keras.layers.MultiHeadAttention(
 num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate)
 Z = attn_layer(Z, value=Z, attention_mask=encoder_pad_mask)
 Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))
 skip = Z
 Z = tf.keras.layers.Dense(n_units, activation="relu")(Z)
 Z = tf.keras.layers.Dense(embed_size)(Z)
 Z = tf.keras.layers.Dropout(dropout_rate)(Z)
 Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

In [ ]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis") # many other tasks are available
result = classifier("The actors were very convincing".)
The result is a Python list containing one dictionary per input text:

In [ ]:
result


In [ ]:
classifier(["I am from India.", "I am from Iraq."])